In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Qualidade de Dados — Camada Bronze do Airbnb Rio de Janeiro
# MAGIC
# MAGIC **Objetivo desta etapa:** avaliar a qualidade dos dados brutos antes de construir a camada Silver,
# MAGIC seguindo as dimensões clássicas de qualidade de dados:
# MAGIC - **Completude**: existem valores nulos/vazios? Em que proporção?
# MAGIC - **Consistência**: os valores seguem um padrão esperado?
# MAGIC - **Unicidade**: existem duplicatas onde não deveria haver?
# MAGIC - **Acurácia**: os valores fazem sentido no contexto?
# MAGIC - **Outliers**: existem valores extremos que podem distorcer análises?
# MAGIC
# MAGIC Os achados aqui **guiam diretamente** as transformações que serão feitas no notebook Bronze → Silver.

# COMMAND ----------

CATALOGO = "mvp_airbnb_rj"
SCHEMA_BRONZE = "bronze"
TABELA_BRONZE = "listings_bronze"

df = spark.table(f"{CATALOGO}.{SCHEMA_BRONZE}.{TABELA_BRONZE}")
print(f"Total de registros: {df.count()}")
print(f"Total de colunas: {len(df.columns)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Visão geral do schema
# MAGIC
# MAGIC Antes de tudo, vamos ver os nomes e tipos de todas as colunas que vieram na inferência automática
# MAGIC (`inferSchema=true`) da camada Bronze. 

# COMMAND ----------

df.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Completude — percentual de nulos por coluna
# MAGIC
# MAGIC Calculamos, para TODAS as colunas, quantos valores nulos/vazios existem e qual o percentual
# MAGIC em relação ao total de linhas. Colunas com altíssimo percentual de nulos (ex: >90%) são
# MAGIC candidatas a serem descartadas na Silver — não fazem sentido manter algo praticamente vazio.

# COMMAND ----------

from pyspark.sql.functions import col, count, when, isnan

total_linhas = df.count()

# Para colunas numéricas, foi checado se há nulos E NaN. Para colunas de texto, checamos nulos e string vazia.
# Foi construída, essa checagem coluna a coluna, pois nem toda coluna aceita isnan() (ex: strings, datas).
resultado_completude = []

for nome_coluna, tipo_coluna in df.dtypes:
    try:
        if tipo_coluna in ("double", "float"):
            qtd_nulos = df.filter(col(nome_coluna).isNull() | isnan(col(nome_coluna))).count()
        else:
            qtd_nulos = df.filter(col(nome_coluna).isNull()).count()
        pct_nulos = round((qtd_nulos / total_linhas) * 100, 2)
        resultado_completude.append((nome_coluna, tipo_coluna, qtd_nulos, pct_nulos))
    except Exception as e:
        resultado_completude.append((nome_coluna, tipo_coluna, None, f"erro: {e}"))

df_completude = spark.createDataFrame(
    resultado_completude,
    ["coluna", "tipo", "qtd_nulos", "pct_nulos"]
).orderBy(col("pct_nulos").desc())

display(df_completude)

# COMMAND ----------

# MAGIC %md
# MAGIC - Quais colunas têm 100% (ou quase) de nulos → candidatas a descarte
# MAGIC - Quais colunas-chave para suas perguntas de negócio (preço, localização, avaliações,
# MAGIC   tipo de imóvel, host) têm nulos significativos → precisarão de tratamento específico na Silver

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Verificação de colunas-chave específicas
# MAGIC
# MAGIC Aqui foi inspecionado mais de perto as colunas que devem alimentar diretamente suas perguntas
# MAGIC de negócio. Os nomes abaixo seguem o padrão comum do Inside Airbnb.

# COMMAND ----------

colunas_chave = [
    "id", "price", "neighbourhood", "neighbourhood_cleansed", "latitude", "longitude",
    "room_type", "property_type", "accommodates", "bedrooms", "bathrooms",
    "minimum_nights", "number_of_reviews", "review_scores_rating",
    "host_is_superhost", "availability_365", "mes_referencia"
]

# Filtra apenas as que realmente existem no DataFrame, para não quebrar o script
colunas_chave_existentes = [c for c in colunas_chave if c in df.columns]
colunas_nao_encontradas = [c for c in colunas_chave if c not in df.columns]

print("Colunas-chave encontradas no dataset:")
print(colunas_chave_existentes)
print("\nColunas-chave NÃO encontradas (confira o nome correto no schema e ajuste a lista acima):")
print(colunas_nao_encontradas)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Unicidade — existem linhas duplicadas?
# MAGIC
# MAGIC Como o dataset é composto por múltiplos arquivos mensais, é esperado que o MESMO imóvel
# MAGIC (mesmo `id`) apareça em vários meses. Duplicata real seria a mesma linha
# MAGIC (mesmo id + mesmo mes_referencia) aparecendo mais de uma vez.

# COMMAND ----------

if "id" in df.columns and "mes_referencia" in df.columns:
    total_linhas_check = df.count()
    total_combinacoes_unicas = df.select("id", "mes_referencia").distinct().count()
    qtd_duplicatas = total_linhas_check - total_combinacoes_unicas

    print(f"Total de linhas: {total_linhas_check}")
    print(f"Combinações únicas de (id + mes_referencia): {total_combinacoes_unicas}")
    print(f"Linhas potencialmente duplicadas (mesmo id no mesmo mês): {qtd_duplicatas}")

    if qtd_duplicatas > 0:
        print("\nExemplos de duplicatas encontradas:")
        display(
            df.groupBy("id", "mes_referencia")
            .count()
            .filter(col("count") > 1)
            .orderBy(col("count").desc())
            .limit(20)
        )
else:
    print("Colunas 'id' e/ou 'mes_referencia' não encontradas — ajuste conforme o schema real.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Consistência e Acurácia — coluna de preço (price)
# MAGIC
# MAGIC No Inside Airbnb, a coluna `price` costuma vir como TEXTO, com símbolo de moeda e vírgula
# MAGIC de milhar (ex: "$1,200.00"), em vez de número. Isso precisa virar decimal na Silver.

# COMMAND ----------

if "price" in df.columns:
    print("Tipo de dado atual da coluna price:", dict(df.dtypes)["price"])
    print("\nAmostra de valores distintos de price:")
    display(df.select("price").distinct().limit(20))
else:
    print("Coluna 'price' não encontrada com esse nome — confira o nome real no schema.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Outliers — valores extremos de preço
# MAGIC
# MAGIC Depois de identificado o formato real da coluna price (célula anterior), esta célula
# MAGIC tenta converter para número e identificar estatísticas básicas + outliers extremos.

# COMMAND ----------

from pyspark.sql.functions import regexp_replace

if "price" in df.columns:
    df_price_numerico = df.withColumn(
        "price_numerico",
        regexp_replace(regexp_replace(col("price").cast("string"), r"\$", ""), r",", "").cast("double")
    )

    df_price_numerico.select("price_numerico").describe().show()

    print("\nTop 10 maiores valores de price (possíveis outliers):")
    display(df_price_numerico.select("id", "price", "price_numerico", "mes_referencia")
            .orderBy(col("price_numerico").desc()).limit(10))

    print("\nRegistros com price igual a zero ou nulo após conversão (possível problema):")
    display(df_price_numerico.filter(
        (col("price_numerico") == 0) | col("price_numerico").isNull()
    ).select("id", "price", "price_numerico", "mes_referencia").limit(10))
else:
    print("Coluna 'price' não encontrada — ajuste o nome conforme o schema real.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Acurácia — coordenadas geográficas fora do Rio de Janeiro
# MAGIC
# MAGIC O Rio de Janeiro está aproximadamente entre as latitudes -23.1 e -22.7, e longitudes
# MAGIC -43.8 e -43.0. Coordenadas fora dessa faixa indicam erro de coleta/scraping.

# COMMAND ----------

if "latitude" in df.columns and "longitude" in df.columns:
    df_coordenadas_invalidas = df.filter(
        (col("latitude") < -23.5) | (col("latitude") > -22.5) |
        (col("longitude") < -44.0) | (col("longitude") > -42.5)
    )
    qtd_invalidas = df_coordenadas_invalidas.count()
    print(f"Registros com coordenadas fora da faixa esperada do Rio de Janeiro: {qtd_invalidas}")
    if qtd_invalidas > 0:
        display(df_coordenadas_invalidas.select("id", "latitude", "longitude", "neighbourhood").limit(10))
else:
    print("Colunas 'latitude'/'longitude' não encontradas com esse nome — confira o schema real.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Consistência — valores categóricos (room_type, host_is_superhost)
# MAGIC
# MAGIC Verificado se colunas categóricas têm um conjunto pequeno e consistente de valores
# MAGIC (ex: "t"/"f" para booleanos, ou variações inesperadas de escrita).

# COMMAND ----------

for coluna_categorica in ["room_type", "property_type", "host_is_superhost"]:
    if coluna_categorica in df.columns:
        print(f"\nValores distintos em '{coluna_categorica}':")
        df.groupBy(coluna_categorica).count().orderBy(col("count").desc()).show(30, truncate=False)
    else:
        print(f"\nColuna '{coluna_categorica}' não encontrada — confira o nome real no schema.")
